# 03. Preprocessing: Feature Selection and Train/Test Split

CRISP-DM phase: Data preparation

## 1. Setup and Data Loading

### 1.1 Importing Libraries

In [1]:
#necessary imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

### 1.2 Loading the Dataset

In [2]:
#loading the dataset
df = pd.read_csv('../data/zenodo_clean.csv')

In [3]:
#shape of the dataset
df.shape

(19251, 85)

## 2. Target Variable

The target is recreated here in the same way as in notebook 02. Each student gets a `level` from the PHQ-9 total: 0 = okay (0 to 4), 1 = mild (5 to 9) and 2 = moderate or worse (10 to 27).

### 2.1 Creating the Target Column

In [4]:
#creating the three levels: 0 = okay, 1 = mild, 2 = moderate or worse
df['level'] = pd.cut(df['phq_total'], bins=[-1, 4, 9, 27], labels=[0, 1, 2]).astype(int)
df['level'].value_counts().sort_index()

level
0    11933
1     6034
2     1284
Name: count, dtype: int64

#### Observation

There are 11,933 okay students (level 0), 6,034 mild students (level 1) and 1,284 moderate or worse students (level 2), the same counts as in notebook 02.

## 3. Feature Selection
### 3.1 Candidate Questions

The candidate pool contains the PHQ-2 questions, the 7 GAD-7 questions, the 7 ISI questions and the 14 PSS questions.

In [5]:
#questions we are allowed to use (phq q3-q9 are left out because they make up the answer)
phq_items = ['phq_q1', 'phq_q2']
gad_items = ['gad_q1', 'gad_q2', 'gad_q3', 'gad_q4', 'gad_q5', 'gad_q6', 'gad_q7']
isi_items = ['isi_q1', 'isi_q2', 'isi_q3', 'isi_q4', 'isi_q5', 'isi_q6', 'isi_q7']
pss_items = ['pss_q1', 'pss_q2', 'pss_q3', 'pss_q4', 'pss_q5', 'pss_q6', 'pss_q7', 'pss_q8', 'pss_q9', 'pss_q10', 'pss_q11', 'pss_q12', 'pss_q13', 'pss_q14']
items = phq_items + gad_items + isi_items + pss_items

In [6]:
#number of candidate questions
len(items)

30

#### Observation

The candidate pool has 30 questions: 2 from the PHQ-9, 7 from the GAD-7, 7 from the ISI and 14 from the PSS-14.

### 3.2 Correlation of Each Question with the PHQ-9 Total

In [7]:
#correlation of each question with the phq-9 total
corr = df[items + ['phq_total']].corr()['phq_total'].drop('phq_total').sort_values(ascending=False)
corr

phq_q2     0.723610
phq_q1     0.710337
gad_q6     0.604204
gad_q2     0.589839
gad_q3     0.586533
gad_q4     0.583413
gad_q1     0.581367
pss_q2     0.570011
pss_q3     0.562709
pss_q1     0.534699
gad_q5     0.525208
gad_q7     0.525156
isi_q1     0.493054
isi_q7     0.472410
isi_q2     0.457171
pss_q14    0.454373
isi_q4     0.448606
pss_q11    0.385191
isi_q5     0.383499
isi_q3     0.328578
pss_q8     0.326087
isi_q6     0.318061
pss_q7     0.257100
pss_q13    0.256488
pss_q9     0.232735
pss_q6     0.200826
pss_q5     0.167794
pss_q10    0.155567
pss_q4     0.113506
pss_q12    0.095765
Name: phq_total, dtype: float64

#### Observation

`phq_q2` (0.72) and `phq_q1` (0.71) are the strongest, followed by the GAD-7 questions (0.53 to 0.60), three PSS questions and the ISI questions. The weakest questions are PSS questions such as `pss_q12` (0.10) and `pss_q4` (0.11).

### 3.3 Selecting the Top 21 Questions

The 21 questions with the highest correlation are kept as the model features. Correlations here are calculated on all students before the split, which is a simplification: the test students had a small say in which questions were picked. The choice is unlikely to change the results, because the 21st question (`pss_q8`, 0.33) and the 22nd (`isi_q6`, 0.32) have very similar correlations.

In [8]:
#picking the top 21 questions by correlation with phq-9
features = corr.head(21).index.tolist()
features

['phq_q2',
 'phq_q1',
 'gad_q6',
 'gad_q2',
 'gad_q3',
 'gad_q4',
 'gad_q1',
 'pss_q2',
 'pss_q3',
 'pss_q1',
 'gad_q5',
 'gad_q7',
 'isi_q1',
 'isi_q7',
 'isi_q2',
 'pss_q14',
 'isi_q4',
 'pss_q11',
 'isi_q5',
 'isi_q3',
 'pss_q8']

### 3.4 Checking for Leakage

The next cell lists any selected feature that is the PHQ-9 total or one of PHQ-9 questions 3 to 9. These make up the target, so the list must be empty.

In [9]:
#checking that no phq-9 total or phq-9 question 3 to 9 is in the features
leak_columns = ['phq_total', 'phq_q3', 'phq_q4', 'phq_q5', 'phq_q6', 'phq_q7', 'phq_q8', 'phq_q9']
print(set(features) & set(leak_columns))

set()


#### Observation

The selected features are the two PHQ-2 questions, seven GAD-7 questions, six ISI questions and six PSS questions. The check prints an empty set, so none of them is the PHQ-9 total or a PHQ-9 question from 3 to 9, and the leakage rule is respected.

## 4. Train/Test Split
### 4.1 Features and Target
`X` contains the 21 selected questions and `y` is the `level` column (0, 1 or 2).

In [10]:
#features (X) and target (y)
X = df[features]
y = df['level']

### 4.2 Splitting the Data

In [11]:
#splitting 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape)
print(X_test.shape)

(15400, 21)
(3851, 21)


#### Observation

The training set has 15,400 students and the test set has 3,851, each with 21 features. This is an 80/20 split of the 19,251 students.

### 4.3 Checking the Class Balance

The split uses `stratify=y`, so each level should have the same share in the training and test sets. The next cells check the percentages and the number of students in each level.

In [12]:
#checking the split kept the same share of each level
print(y_train.value_counts(normalize=True).sort_index() * 100)
print(y_test.value_counts(normalize=True).sort_index() * 100)

level
0    61.987013
1    31.344156
2     6.668831
Name: proportion, dtype: float64
level
0    61.983900
1    31.342508
2     6.673591
Name: proportion, dtype: float64


In [13]:
#number of students in each level in each set
print(y_train.value_counts().sort_index())
print(y_test.value_counts().sort_index())

level
0    9546
1    4827
2    1027
Name: count, dtype: int64
level
0    2387
1    1207
2     257
Name: count, dtype: int64


#### Observation

Stratification worked: the shares of level 0, 1 and 2 are 62.0%, 31.3% and 6.7% in both the training set and the test set, matching the full dataset. The training set contains 9,546, 4,827 and 1,027 students in levels 0, 1 and 2, and the test set contains 2,387, 1,207 and 257. Because the test set has only 257 level 2 students, each one missed or caught moves the recall of level 2 by about 0.4 percentage points, so the test results carry some uncertainty.

## 5. Saving the Split

The training and test sets are saved with their `level` labels in the `data` folder. The base model and all later models load these exact files, so any difference in results comes from the model and not from a different random split. This makes the comparison fair.

In [14]:
#saving the split so every model uses the same data
pd.concat([X_train, y_train], axis=1).to_csv('../data/train.csv', index=False)
pd.concat([X_test, y_test], axis=1).to_csv('../data/test.csv', index=False)

In [15]:
#checking the saved files
print(pd.read_csv('../data/train.csv').shape)
print(pd.read_csv('../data/test.csv').shape)

(15400, 22)
(3851, 22)


In [16]:
#checking the label column and its levels in the saved training file
train_check = pd.read_csv('../data/train.csv')
print(train_check.columns[-1])
print(train_check['level'].value_counts().sort_index())

level
level
0    9546
1    4827
2    1027
Name: count, dtype: int64


#### Observation

`train.csv` has 15,400 rows and 22 columns, and `test.csv` has 3,851 rows and 22 columns. The 22 columns are the 21 questions plus the `level` label, and the last column of the saved training file is `level`. The row counts match the split in Section 4.2, so the files were written correctly. The next notebook, `LogisticRegression_Base.ipynb`, trains the base model on these files.